In [1]:
# =========================================================
# PS S6E4 - exp_20260410_034_dao_xgb_pair_te_core_min
# minimum reproduction of Dao core:
# raw base + pairwise 171->accepted + pairwise multiclass TE + XGB
# no pseudo-label / no post-hoc optimization / no formula-logit
# =========================================================

## Import / config

In [2]:
import os
import gc
import json
import time
import random
from itertools import combinations

import numpy as np
import pandas as pd
import xgboost as xgb
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
class CFG:
    EXP_ID = "exp_20260410_034_dao_xgb_pair_te_core_min"
    SEED = 42
    N_SPLITS = 5
    INNER_TE_CV = 5

    TARGET = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    ORIG_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    # pair acceptance rule:
    # keep only if unique count <= half of combined rows
    # this matches the Dao snippet's filtering logic
    PAIR_UNIQUE_RATIO_MAX = 0.5

    XGB_PARAMS = {
        "n_estimators": 20000,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 2.333941903991847,
        "subsample": 0.9766412297733108,
        "colsample_bytree": 0.535324419516146,
        "gamma": 4.258489082295074,
        "reg_alpha": 4.082875850185249e-08,
        "reg_lambda": 0.00013528868091784412,
        "objective": "multi:softprob",
        "num_class": 3,
        "tree_method": "hist",
        "enable_categorical": False,  # after TE, everything passed as numeric
        "eval_metric": "mlogloss",
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": 0,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }

## Utils

In [3]:
# ---------------------------------------------------------
# Utils
# ---------------------------------------------------------
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def save_json(obj, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def rankdata_1d(a: np.ndarray) -> np.ndarray:
    return pd.Series(a).rank(method="average").to_numpy()

def mean_rank_corr_multiclass(a: np.ndarray, b: np.ndarray) -> float:
    vals = []
    for c in range(a.shape[1]):
        ra = rankdata_1d(a[:, c])
        rb = rankdata_1d(b[:, c])
        vals.append(np.corrcoef(ra, rb)[0, 1])
    return float(np.mean(vals))

def mean_raw_corr_multiclass(a: np.ndarray, b: np.ndarray) -> float:
    vals = []
    for c in range(a.shape[1]):
        vals.append(np.corrcoef(a[:, c], b[:, c])[0, 1])
    return float(np.mean(vals))

def build_pairwise_features(train_df: pd.DataFrame, test_df: pd.DataFrame, all_cols: list[str]):
    train_out = train_df.copy()
    test_out = test_df.copy()

    te_columns = []
    pair_meta = []

    n_train = len(train_out)

    for c1, c2 in combinations(all_cols, 2):
        name = f"{c1}|{c2}"

        tr_s = train_out[c1].astype(str) + "_" + train_out[c2].astype(str)
        te_s = test_out[c1].astype(str) + "_" + test_out[c2].astype(str)

        combined = pd.concat([tr_s, te_s], axis=0, ignore_index=True)
        codes, uniques = pd.factorize(combined)

        nunique = pd.Series(codes).nunique()
        total_n = len(codes)

        # Dao snippet rule:
        # if nunique > len(codes) // 2: continue
        if nunique > total_n * CFG.PAIR_UNIQUE_RATIO_MAX:
            continue

        train_out[name] = codes[:n_train].astype("int32")
        test_out[name] = codes[n_train:].astype("int32")
        te_columns.append(name)

        pair_meta.append({
            "pair_name": name,
            "col1": c1,
            "col2": c2,
            "nunique_total": int(nunique),
            "n_rows_total": int(total_n),
            "ratio_unique": float(nunique / total_n),
        })

    return train_out, test_out, te_columns, pd.DataFrame(pair_meta)

def apply_pair_te(X_tr_pair, y_tr, X_va_pair, X_te_pair, cols):
    enc = TargetEncoder(
        target_type="multiclass",
        cv=CFG.INNER_TE_CV,
        random_state=CFG.SEED
    )

    tr_enc = pd.DataFrame(
        enc.fit_transform(X_tr_pair[cols], y_tr),
        index=X_tr_pair.index
    )
    va_enc = pd.DataFrame(
        enc.transform(X_va_pair[cols]),
        index=X_va_pair.index
    )
    te_enc = pd.DataFrame(
        enc.transform(X_te_pair[cols]),
        index=X_te_pair.index
    )

    tr_enc.columns = [f"TE_{col}_cls{k}" for col in cols for k in range(3)]
    va_enc.columns = tr_enc.columns
    te_enc.columns = tr_enc.columns

    return tr_enc.reset_index(drop=True), va_enc.reset_index(drop=True), te_enc.reset_index(drop=True)

## Load data & FE

In [4]:
# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
seed_everything(CFG.SEED)
ensure_dir(CFG.OUTPUT_DIR)

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
orig = pd.read_csv(CFG.ORIG_PATH)  # loaded only for consistency / later comparison; unused in 034 core-min

train[CFG.TARGET] = train[CFG.TARGET].map(CFG.TARGET_MAP)

num_features = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

cat_features = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Mulching_Used",
    "Region",
]

all_base_cols = num_features + cat_features

# raw base only
train_eng = train[[CFG.ID_COL] + all_base_cols + [CFG.TARGET]].copy()
test_eng = test[[CFG.ID_COL] + all_base_cols].copy()

# pairwise interactions
train_eng, test_eng, te_columns, pair_meta_df = build_pairwise_features(
    train_eng, test_eng, all_base_cols
)

# base features = raw 19 only
base_features = all_base_cols.copy()

X_base = train_eng[base_features].copy()
X_tbase = test_eng[base_features].copy()
X_pair = train_eng[te_columns].copy()
X_tpair = test_eng[te_columns].copy()
y_full = train_eng[CFG.TARGET].values

# raw base must be numeric for XGB after concat with TE
# category columns are factorized jointly across train/test
for c in cat_features:
    combined = pd.concat([X_base[c].astype(str), X_tbase[c].astype(str)], axis=0, ignore_index=True)
    codes, uniques = pd.factorize(combined)
    X_base[c] = codes[:len(X_base)].astype("int32")
    X_tbase[c] = codes[len(X_base):].astype("int32")

for c in num_features:
    med = X_base[c].median()
    X_base[c] = X_base[c].fillna(med).astype("float32")
    X_tbase[c] = X_tbase[c].fillna(med).astype("float32")

for c in cat_features:
    X_base[c] = X_base[c].astype("float32")
    X_tbase[c] = X_tbase[c].astype("float32")

print(f"Base features: {len(base_features)}")
print(f"Raw pair candidates: {len(list(combinations(all_base_cols, 2)))}")
print(f"Accepted pair features: {len(te_columns)}")

pair_meta_df.to_csv(f"{CFG.OUTPUT_DIR}/pair_feature_meta.csv", index=False)

/tmp/ipykernel_24/4052823925.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_out[name] = codes[:n_train].astype("int32")
/tmp/ipykernel_24/4052823925.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_out[name] = codes[n_train:].astype("int32")
/tmp/ipykernel_24/4052823925.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragme

Base features: 19
Raw pair candidates: 171
Accepted pair features: 135


## CV train

In [5]:
# ---------------------------------------------------------
# CV training
# ---------------------------------------------------------
skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED
)

oof_probs = np.zeros((len(X_base), 3), dtype=np.float32)
test_probs = np.zeros((len(X_tbase), 3), dtype=np.float32)

fold_scores = []
best_iterations = []
feature_importance_list = []

sample_weights_all = compute_sample_weight("balanced", y_full)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_base, y_full), 1):
    print(f"\n===== Fold {fold}/{CFG.N_SPLITS} =====")

    X_tr_base = X_base.iloc[tr_idx].reset_index(drop=True)
    X_va_base = X_base.iloc[va_idx].reset_index(drop=True)
    X_te_base = X_tbase.reset_index(drop=True)

    y_tr = y_full[tr_idx]
    y_va = y_full[va_idx]

    X_tr_pair = X_pair.iloc[tr_idx].reset_index(drop=True)
    X_va_pair = X_pair.iloc[va_idx].reset_index(drop=True)
    X_te_pair = X_tpair.reset_index(drop=True)

    tr_te, va_te, te_te = apply_pair_te(
        X_tr_pair, y_tr, X_va_pair, X_te_pair, te_columns
    )

    Xt = pd.concat([X_tr_base, tr_te], axis=1).astype("float32")
    Xv = pd.concat([X_va_base, va_te], axis=1).astype("float32")
    Xe = pd.concat([X_te_base, te_te], axis=1).astype("float32")

    sw_tr = compute_sample_weight("balanced", y_tr)

    model = xgb.XGBClassifier(
        **CFG.XGB_PARAMS,
        early_stopping_rounds=300
    )

    model.fit(
        Xt,
        y_tr,
        sample_weight=sw_tr,
        eval_set=[(Xv, y_va)],
        verbose=500
    )

    oof_probs[va_idx] = model.predict_proba(Xv)
    test_probs += model.predict_proba(Xe) / CFG.N_SPLITS

    fold_score = balanced_accuracy_score(y_va, np.argmax(oof_probs[va_idx], axis=1))
    fold_scores.append(float(fold_score))
    best_iterations.append(int(getattr(model, "best_iteration", model.n_estimators)))

    booster_score = model.get_booster().get_score(importance_type="gain")
    feature_importance_list.append(booster_score)

    print(f"Fold {fold} BA: {fold_score:.6f}")

    del Xt, Xv, Xe, tr_te, va_te, te_te, model
    gc.collect()

cv_score = float(balanced_accuracy_score(y_full, np.argmax(oof_probs, axis=1)))
print(f"\nOOF Balanced Accuracy: {cv_score:.6f}")


===== Fold 1/5 =====
[0]	validation_0-mlogloss:1.06555
[500]	validation_0-mlogloss:0.06063
[1000]	validation_0-mlogloss:0.05361
[1500]	validation_0-mlogloss:0.05200
[2000]	validation_0-mlogloss:0.05165
[2500]	validation_0-mlogloss:0.05157
[3000]	validation_0-mlogloss:0.05151
[3500]	validation_0-mlogloss:0.05149
[4000]	validation_0-mlogloss:0.05147
[4500]	validation_0-mlogloss:0.05145
[5000]	validation_0-mlogloss:0.05144
[5500]	validation_0-mlogloss:0.05142
[5822]	validation_0-mlogloss:0.05140
Fold 1 BA: 0.976574

===== Fold 2/5 =====
[0]	validation_0-mlogloss:1.06566
[500]	validation_0-mlogloss:0.06175
[1000]	validation_0-mlogloss:0.05479
[1500]	validation_0-mlogloss:0.05308
[2000]	validation_0-mlogloss:0.05266
[2500]	validation_0-mlogloss:0.05256
[3000]	validation_0-mlogloss:0.05250
[3500]	validation_0-mlogloss:0.05248
[4000]	validation_0-mlogloss:0.05245
[4500]	validation_0-mlogloss:0.05243
[5000]	validation_0-mlogloss:0.05241
[5500]	validation_0-mlogloss:0.05240
[6000]	validation_0

## Save outputs

In [6]:
# ---------------------------------------------------------
# Save outputs
# ---------------------------------------------------------
np.save(f"{CFG.OUTPUT_DIR}/oof_dao_xgb_pair_te_core_min_proba.npy", oof_probs)
np.save(f"{CFG.OUTPUT_DIR}/pred_dao_xgb_pair_te_core_min_proba.npy", test_probs)

submission = pd.DataFrame({
    CFG.ID_COL: test[CFG.ID_COL],
    CFG.TARGET: pd.Series(np.argmax(test_probs, axis=1)).map(CFG.INV_TARGET_MAP)
})
submission.to_csv(f"{CFG.OUTPUT_DIR}/submission_dao_xgb_pair_te_core_min.csv", index=False)

# feature columns
feature_columns = pd.DataFrame({
    "feature_name": list(X_base.columns) + [f"TE_{col}_cls{k}" for col in te_columns for k in range(3)],
    "feature_group": (["base_raw"] * len(X_base.columns)) + (["pair_te"] * (len(te_columns) * 3))
})
feature_columns.to_csv(f"{CFG.OUTPUT_DIR}/feature_columns.csv", index=False)

# mean feature importance
all_feats = set()
for d in feature_importance_list:
    all_feats.update(d.keys())

rows = []
for feat in sorted(all_feats):
    vals = [imp.get(feat, 0.0) for imp in feature_importance_list]
    rows.append({
        "feature_name": feat,
        "importance_gain_mean": float(np.mean(vals)),
        "importance_gain_std": float(np.std(vals)),
    })

feature_importance_df = pd.DataFrame(rows).sort_values(
    "importance_gain_mean", ascending=False
)
feature_importance_df.to_csv(f"{CFG.OUTPUT_DIR}/feature_importance.csv", index=False)

# cv json
cv_result = {
    "exp_id": CFG.EXP_ID,
    "model": "XGBoost",
    "objective": "multi:softprob",
    "metric": "balanced_accuracy_score",
    "seed": CFG.SEED,
    "fold_type": "StratifiedKFold",
    "n_splits": CFG.N_SPLITS,
    "inner_te_cv": CFG.INNER_TE_CV,
    "base_feature_count": len(base_features),
    "pair_candidate_count": len(list(combinations(all_base_cols, 2))),
    "pair_accepted_count": len(te_columns),
    "fold_scores": fold_scores,
    "cv_score_raw": cv_score,
    "best_iterations": best_iterations,
    "xgb_params": CFG.XGB_PARAMS,
    "notes": {
        "pseudo_label": False,
        "posthoc_optimization": False,
        "formula_logit": False,
        "original_merge": False,
        "digit_features": False,
        "ordered_te": False,
        "pair_te_only": True
    }
}
save_json(cv_result, f"{CFG.OUTPUT_DIR}/cv_result.json")

print("\nSaved files:")
for fn in [
    "oof_dao_xgb_pair_te_core_min_proba.npy",
    "pred_dao_xgb_pair_te_core_min_proba.npy",
    "submission_dao_xgb_pair_te_core_min.csv",
    "feature_columns.csv",
    "feature_importance.csv",
    "pair_feature_meta.csv",
    "cv_result.json",
]:
    print("-", f"{CFG.OUTPUT_DIR}/{fn}")


Saved files:
- /kaggle/working/exp_20260410_034_dao_xgb_pair_te_core_min/oof_dao_xgb_pair_te_core_min_proba.npy
- /kaggle/working/exp_20260410_034_dao_xgb_pair_te_core_min/pred_dao_xgb_pair_te_core_min_proba.npy
- /kaggle/working/exp_20260410_034_dao_xgb_pair_te_core_min/submission_dao_xgb_pair_te_core_min.csv
- /kaggle/working/exp_20260410_034_dao_xgb_pair_te_core_min/feature_columns.csv
- /kaggle/working/exp_20260410_034_dao_xgb_pair_te_core_min/feature_importance.csv
- /kaggle/working/exp_20260410_034_dao_xgb_pair_te_core_min/pair_feature_meta.csv
- /kaggle/working/exp_20260410_034_dao_xgb_pair_te_core_min/cv_result.json
